In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")
import pickle
import os

In [2]:
vitamin_dReport = pd.read_csv(r"C:\Users\direk\OneDrive\Desktop\data_diseasePredictor\vitamin_d_report.csv")
vitamin_dReport = vitamin_dReport.drop("risk_label", axis=1)
vitamin_dReport.head()

,vitamin_d
0,28.68
1,64.74
2,11.71
3,23.78
4,25.79


In [3]:
def vitamin_d_defeciency(v):
    if v < 10:
        return "Critical"
    elif v < 20:
        return "High"
    elif v < 30:
        return "Medium"
    else:
        return "Low"

vitamin_dReport["vitamin_d_deficency"] = vitamin_dReport["vitamin_d"].apply(vitamin_d_defeciency)
vitamin_dReport.head()

,vitamin_d,vitamin_d_deficency
0,28.68,Medium
1,64.74,Low
2,11.71,High
3,23.78,Medium
4,25.79,Medium


In [4]:
LABEL_MAPPING = {
    "Low": "low",
    "Normal": "moderate",
    "Medium": "moderate",
    "High": "high",
    "Critical": "critical"
}

NUM_MAPPING = {
    "low": 0,
    "moderate": 1,
    "high": 2,
    "critical": 3
}

In [5]:
vitamin_dReport["vitamin_d_deficency"] = vitamin_dReport["vitamin_d_deficency"].map(LABEL_MAPPING)
vitamin_dReport.head()

,vitamin_d,vitamin_d_deficency
0,28.68,moderate
1,64.74,low
2,11.71,high
3,23.78,moderate
4,25.79,moderate


In [6]:
vitamin_dReport["vitamin_d_deficency_num"] = vitamin_dReport["vitamin_d_deficency"].map(NUM_MAPPING)
vitamin_dReport.head()

,vitamin_d,vitamin_d_deficency,vitamin_d_deficency_num
0,28.68,moderate,1
1,64.74,low,0
2,11.71,high,2
3,23.78,moderate,1
4,25.79,moderate,1


In [15]:
"""Auto detect classes and print report - works for any number of classes"""
CLASS_NAMES = {0: "low", 1: "moderate", 2: "high", 3: "critical"}

def print_report(y_test, y_pred, model_name):
    # automatically finds which classes exist in test + predictions
    existing_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
    existing_names = [CLASS_NAMES[i] for i in existing_labels]

    print("=" * 40)
    print(f"{model_name} MODEL ACCURACY")
    print("=" * 40)
    print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print()
    print("=" * 40)
    print(f"{model_name} CLASSIFICATION REPORT")
    print("=" * 40)
    print(classification_report(
        y_test, y_pred,
        labels=existing_labels,
        target_names=existing_names
    ))

In [16]:
"""Preparing data for ML prediction"""
feature_cols=["vitamin_d"]
X = vitamin_dReport[feature_cols]
y = vitamin_dReport["vitamin_d_deficency_num"]

In [17]:
"""train test split"""
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=31)

In [20]:
model_vitaminD = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)
model_vitaminD.fit(X_train, y_train)
y_predict_vitaminD = model_vitaminD.predict(X_test)

In [21]:
print_report(y_test, y_predict_vitaminD, "VITAMIN D DEFECIENCY")

VITAMIN D DEFECIENCY MODEL ACCURACY
Accuracy: 99.00%

VITAMIN D DEFECIENCY CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       1.00      1.00      1.00        48
    moderate       1.00      0.97      0.98        32
        high       0.93      1.00      0.96        13
    critical       1.00      1.00      1.00         7

    accuracy                           0.99       100
   macro avg       0.98      0.99      0.99       100
weighted avg       0.99      0.99      0.99       100



In [22]:
"""Saving the model in pkl file"""
save_path = r"C:\Users\direk\Disease_risk_predictor_-3\ml_models\xgboost"
os.makedirs(save_path, exist_ok=True)
with open(os.path.join(save_path, "vitamin_d.pkl"), "wb") as f:
    pickle.dump(model_vitaminD, f)
    print("vitamin_d.pkl ")

vitamin_d.pkl 
